# Dawn to Dusk — Interactive Dashboard
___
**COMP 4304\
Muneeb Mennad\
202175220**

In [1]:
##########################################################
# IMPORTS
##########################################################
import pandas as pd
import numpy as np
import matplotlib as mpl
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap
import ipywidgets as widgets
from IPython.display import display, clear_output

In [2]:
####################################################################################
# DESIGN
####################################################################################
BG         = '#000000'
TEXT       = '#f4ecd8'
TEXT_MUTED = '#8a8578'
GRID       = '#1e2332'
ACCENT     = '#f2a623'

PREDAWN  = '#3d2c5c'
DAWN     = '#e85d24'
MORNING  = '#f2a623'
MIDDAY   = '#fcde5a'
AFTNOON  = '#e8a23c'
DUSK     = '#8b5cf6'
NIGHT    = '#3b5998'
DEEPNITE = '#1e2a4a'


def hour_gradient(n=24):
    stops = [
        (0,  DEEPNITE),
        (4,  PREDAWN),
        (6,  DAWN),
        (9,  MORNING),
        (12, MIDDAY),
        (15, AFTNOON),
        (18, DUSK),
        (20, NIGHT),
        (23, DEEPNITE),
    ]
    positions = [s[0] / 23 for s in stops]
    colors    = [s[1] for s in stops]
    cmap = LinearSegmentedColormap.from_list('sun_moon', list(zip(positions, colors)), N=256)
    return [cmap(h / 23) for h in range(n)]


HOUR_COLORS    = hour_gradient(24)
SUMMER_COLORS  = [DAWN, MORNING, MIDDAY, AFTNOON, '#d97442']
WINTER_COLORS  = [NIGHT, '#5a7bb8', DUSK, '#6b5fa8', PREDAWN]
SPECIES_COLORS = [DAWN, MORNING, MIDDAY, DUSK, NIGHT, '#d97442']

SEASON_MAP = {
    'All Seasons':      list(range(1, 13)),
    'Spring (Mar–May)': [3, 4, 5],
    'Summer (Jun–Aug)': [6, 7, 8],
    'Fall (Sep–Nov)':   [9, 10, 11],
    'Winter (Dec–Feb)': [12, 1, 2],
}

def pick_font():
    """Prefer JetBrains Mono, fall back to DejaVu Sans Mono."""
    available = {f.name for f in font_manager.fontManager.ttflist}
    for candidate in ('JetBrains Mono', 'JetBrainsMono', 'DejaVu Sans Mono'):
        if candidate in available:
            return candidate
    return 'monospace'


FONT = pick_font()

mpl.rcParams.update({
    'figure.facecolor':  BG,
    'savefig.facecolor': BG,
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

print(f'libraries loaded · font: {FONT}')

libraries loaded · font: JetBrains Mono


In [3]:
df = pd.read_csv('birds.csv')
df['date'] = pd.to_datetime(df['OBSERVATION DATE'])
df['month'] = df['date'].dt.month
df['hour'] = pd.to_datetime(
    df['TIME OBSERVATIONS STARTED'], format='%H:%M:%S', errors='coerce'
).dt.hour
df['county_short'] = df['COUNTY'].str.split('-').str[0].str.strip()

# precompute for widgets
top_species_list = df['COMMON NAME'].value_counts()
top_species_list = top_species_list[top_species_list >= 300].index.tolist()
county_list = ['All Counties'] + sorted(df['county_short'].dropna().unique().tolist())

print(f'Loaded {len(df):,} observations · {df["COMMON NAME"].nunique()} species · {len(county_list)-1} counties')

Loaded 308,370 observations · 300 species · 11 counties


In [4]:
########################################################################
# FILTER FUNCTIONS
########################################################################
def filter_data(season, county, hour_min, hour_max, species=None):
    months = SEASON_MAP[season]
    return _apply_filters(months, county, hour_min, hour_max, species)


def filter_fixed_season(months, county, hour_min, hour_max, species=None):
    return _apply_filters(months, county, hour_min, hour_max, species)


def _apply_filters(months, county, hour_min, hour_max, species):
    mask = df['month'].isin(months)
    if county != 'All Counties':
        mask &= df['county_short'] == county
    mask &= df['hour'].between(hour_min, hour_max)
    if species:
        mask &= df['COMMON NAME'].isin(species)
    return df[mask]

In [5]:
########################################################################
# WIDGETS
########################################################################
style = {'description_width': '80px'}

w_season = widgets.Dropdown(
    options=list(SEASON_MAP.keys()),
    value='All Seasons',
    description='Season:',
    style=style,
)

w_county = widgets.Dropdown(
    options=county_list,
    value='All Counties',
    description='County:',
    style=style,
)

w_hours = widgets.IntRangeSlider(
    value=[0, 23],
    min=0, max=23, step=1,
    description='Hours:',
    style=style,
    continuous_update=False,
    readout_format='02d',
)

w_species = widgets.SelectMultiple(
    options=top_species_list,
    value=['American Herring Gull', 'American Robin',
           "Lincoln's Sparrow", 'Northern Yellow Warbler'],
    description='Species:',
    style=style,
    rows=8,
)

out = widgets.Output()

In [6]:
########################################################################
# DASHB
########################################################################
def bucket_count(df_season, lo, hi):
    return ((df_season['hour'] >= lo) & (df_season['hour'] < hi)).sum()


def scatter_sizes(counts, max_val, max_size=18000):
    return max_size * (counts / max_val)


def draw_dashboard(season, county, hour_range, species_selected):
    hour_min, hour_max = hour_range
    species_list = list(species_selected) if species_selected else None

    dff_clock   = filter_data(season, county, hour_min, hour_max, species_list)
    dff_summer  = filter_fixed_season([6, 7, 8],  county, hour_min, hour_max, species_list)
    dff_winter  = filter_fixed_season([12, 1, 2], county, hour_min, hour_max, species_list)
    dff_species = filter_data(season, county, hour_min, hour_max)

    n_obs       = len(dff_clock)
    n_species   = dff_clock['COMMON NAME'].nunique()
    n_observers = dff_clock['OBSERVER ID'].nunique()
    peak_hour   = int(dff_clock['hour'].mode().iloc[0]) if len(dff_clock) > 0 else 0

    fig = plt.figure(figsize=(18, 22), facecolor=BG)

    fig.text(0.04, 0.975, 'dawn to dusk',
             ha='left', va='top', color=TEXT,
             fontsize=36, fontweight='bold', family=FONT)
    fig.text(0.04, 0.955, "explore when and where newfoundland's birds come alive",
             ha='left', va='top', color=TEXT_MUTED,
             fontsize=14, family=FONT)

    stats_y = 0.938
    stat_items = [
        ('observations', f'{n_obs:,}'),
        ('species',      f'{n_species}'),
        ('observers',    f'{n_observers:,}'),
        ('peak hour',    f'{peak_hour:02d}:00'),
    ]
    for i, (label, val) in enumerate(stat_items):
        x = 0.04 + i * 0.22
        fig.text(x, stats_y, val, ha='left', va='top',
                 color=ACCENT, fontsize=22, fontweight='bold', family=FONT)
        fig.text(x, stats_y - 0.020, label.upper(), ha='left', va='top',
                 color=TEXT_MUTED, fontsize=9, family=FONT)

    fig.add_artist(plt.Line2D([0.04, 0.96], [0.905, 0.905],
                              color=GRID, linewidth=0.8))

    fig.text(0.04, 0.895, 'THE BIRDING CLOCK',
             ha='left', va='top', color=TEXT,
             fontsize=15, fontweight='bold', family=FONT)
    fig.text(0.04, 0.880, 'observation volume by hour of day',
             ha='left', va='top', color=TEXT_MUTED, fontsize=10, family=FONT)

    ax1 = fig.add_axes([0.04, 0.56, 0.40, 0.30], projection='polar', facecolor=BG)

    hourly = dff_clock.groupby('hour').size().reindex(range(24), fill_value=0)
    angles = np.array([2 * np.pi * h / 24 for h in range(24)])
    bar_width = 2 * np.pi / 24 * 0.88

    ax1.bar(angles, hourly.values, width=bar_width, color=HOUR_COLORS,
            edgecolor=BG, linewidth=1.2, align='center', zorder=3)
    ax1.set_theta_zero_location('N')
    ax1.set_theta_direction(-1)
    ax1.set_xticks(angles[::3])
    ax1.set_xticklabels([f'{h:02d}:00' for h in range(0, 24, 3)],
                        fontsize=10, color=TEXT_MUTED)
    ax1.grid(color=GRID, linewidth=0.5, alpha=0.8)
    ax1.spines['polar'].set_color(GRID)
    ax1.spines['polar'].set_linewidth(0.5)
    ax1.tick_params(axis='y', labelsize=8, labelcolor=TEXT_MUTED)
    ax1.set_rlabel_position(292)

    fig.text(0.52, 0.895, 'THE BIRDING DAY COMPRESSES',
             ha='left', va='top', color=TEXT,
             fontsize=15, fontweight='bold', family=FONT)
    fig.text(0.52, 0.880, 'summer vs winter observation volume by time bucket',
             ha='left', va='top', color=TEXT_MUTED, fontsize=10, family=FONT)

    ax2 = fig.add_axes([0.52, 0.56, 0.44, 0.30], facecolor=BG)

    buckets = [
        ('early morning', '05–08', (5, 8)),
        ('mid morning',   '08–11', (8, 11)),
        ('midday',        '11–14', (11, 14)),
        ('afternoon',     '14–17', (14, 17)),
        ('evening',       '17–20', (17, 20)),
    ]

    summer_counts = np.array([bucket_count(dff_summer, lo, hi) for _, _, (lo, hi) in buckets])
    winter_counts = np.array([bucket_count(dff_winter, lo, hi) for _, _, (lo, hi) in buckets])

    max_val = max(summer_counts.max(), winter_counts.max(), 1)
    x_positions = np.arange(len(buckets))
    y_summer = 1.2
    y_winter = -1.2

    ax2.scatter(x_positions, [y_summer] * len(buckets),
                s=scatter_sizes(summer_counts, max_val), c=SUMMER_COLORS,
                alpha=0.9, zorder=3, edgecolors='none')
    ax2.scatter(x_positions, [y_winter] * len(buckets),
                s=scatter_sizes(winter_counts, max_val), c=WINTER_COLORS,
                alpha=0.9, zorder=3, edgecolors='none')

    for i, (label, time_range, _) in enumerate(buckets):
        s_off = np.sqrt(scatter_sizes(summer_counts, max_val)[i]) / 130 + 0.25
        w_off = np.sqrt(scatter_sizes(winter_counts, max_val)[i]) / 130 + 0.25

        ax2.text(i, y_summer + s_off, f'{summer_counts[i]:,}',
                 ha='center', va='bottom', color=TEXT, fontsize=11, family=FONT)
        ax2.text(i, y_winter - w_off, f'{winter_counts[i]:,}',
                 ha='center', va='top', color=TEXT, fontsize=11, family=FONT)
        ax2.text(i, 0.08, time_range,
                 ha='center', va='center', color=TEXT, fontsize=12,
                 fontweight='bold', family=FONT)
        ax2.text(i, -0.18, label,
                 ha='center', va='center', color=TEXT_MUTED, fontsize=9, family=FONT)

    ax2.text(-0.9, y_summer, 'summer',
             ha='center', va='center', color=TEXT, fontsize=16,
             fontweight='bold', family=FONT)
    ax2.text(-0.9, y_summer - 0.30, 'jun · jul · aug',
             ha='center', va='center', color=TEXT_MUTED, fontsize=9, family=FONT)
    ax2.text(-0.9, y_winter, 'winter',
             ha='center', va='center', color=TEXT, fontsize=16,
             fontweight='bold', family=FONT)
    ax2.text(-0.9, y_winter - 0.30, 'dec · jan · feb',
             ha='center', va='center', color=TEXT_MUTED, fontsize=9, family=FONT)

    ax2.set_xlim(-1.8, len(buckets) - 0.3)
    ax2.set_ylim(-2.5, 2.5)
    ax2.axis('off')

    fig.add_artist(plt.Line2D([0.04, 0.96], [0.535, 0.535],
                              color=GRID, linewidth=0.8))

    fig.text(0.04, 0.52, 'A DAY IN THE LIFE',
             ha='left', va='top', color=TEXT,
             fontsize=15, fontweight='bold', family=FONT)
    fig.text(0.04, 0.505, 'different birds own different hours — relative activity for selected species',
             ha='left', va='top', color=TEXT_MUTED, fontsize=10, family=FONT)

    ax3 = fig.add_axes([0.08, 0.08, 0.86, 0.40], facecolor=BG)

    if species_list:
        hours_list = list(range(hour_min, hour_max + 1))
        total_per_hour = dff_species.groupby('hour').size().reindex(hours_list, fill_value=0)

        for i, sp in enumerate(species_list[:6]):
            sp_data = dff_species[dff_species['COMMON NAME'] == sp]
            sp_hourly = sp_data.groupby('hour').size().reindex(hours_list, fill_value=0)
            rate = sp_hourly / total_per_hour.replace(0, np.nan)
            rate = rate.fillna(0)
            rel = rate / rate.max() if rate.max() > 0 else rate

            color = SPECIES_COLORS[i % len(SPECIES_COLORS)]
            ax3.fill_between(hours_list, rel, alpha=0.15, color=color)
            ax3.plot(hours_list, rel, color=color, linewidth=2.5,
                     label=sp.lower(), zorder=3)
            if rate.max() > 0:
                peak_h = int(rate.idxmax())
                ax3.plot(peak_h, 1.0, 'o', color=color, markersize=9,
                         markeredgecolor=BG, markeredgewidth=2, zorder=4)
    else:
        ax3.text(0.5, 0.5, 'select species above to compare',
                 ha='center', va='center', transform=ax3.transAxes,
                 color=TEXT_MUTED, fontsize=14)

    ax3.set_xlim(hour_min, hour_max)
    ax3.set_ylim(0, 1.18)
    ax3.set_xticks(range(hour_min, hour_max + 1, 2))
    ax3.set_xticklabels([f'{h:02d}' for h in range(hour_min, hour_max + 1, 2)])
    ax3.set_xlabel('hour of day', color=TEXT_MUTED, fontsize=12, labelpad=10)
    ax3.set_ylabel('relative activity', color=TEXT_MUTED, fontsize=12, labelpad=10)
    ax3.tick_params(colors=TEXT_MUTED, labelsize=10)
    ax3.grid(axis='y', color=GRID, linewidth=0.5, alpha=0.7)
    ax3.set_axisbelow(True)
    for spine in ax3.spines.values():
        spine.set_color(GRID)
        spine.set_linewidth(0.5)
    if species_list:
        ax3.legend(loc='upper right', frameon=False, fontsize=11,
                   labelcolor=TEXT, handlelength=1.5)

    fig.add_artist(plt.Line2D([0.04, 0.96], [0.04, 0.04],
                              color=GRID, linewidth=0.8))
    fig.text(0.04, 0.030, 'Muneeb Mennad',
             ha='left', va='top', color=TEXT_MUTED, fontsize=9)
    fig.text(0.96, 0.030, 'Memorial University of Newfoundland · COMP 4304',
             ha='right', va='top', color=TEXT_MUTED, fontsize=9)

    plt.show()

In [7]:
def on_change(change):
    with out:
        clear_output(wait=True)
        draw_dashboard(
            w_season.value,
            w_county.value,
            w_hours.value,
            w_species.value,
        )
        
w_season.observe(on_change, names='value')
w_county.observe(on_change, names='value')
w_hours.observe(on_change, names='value')
w_species.observe(on_change, names='value')

In [8]:
controls_top = widgets.HBox(
    [w_season, w_county, w_hours],
    layout=widgets.Layout(
        justify_content='flex-start',
        gap='20px',
        padding='10px 0',
    ),
)

controls_bot = widgets.HBox(
    [w_species],
    layout=widgets.Layout(
        padding='0 0 10px 0',
    ),
)

dashboard = widgets.VBox(
    [controls_top, controls_bot, out],
    layout=widgets.Layout(padding='10px'),
)

display(dashboard)

on_change(None)